# 02 — Ablations : gains locaux, gains publics

Ce notebook trace l'historique des versions (tableau complet dans
`results/versions.md`) et la calibration locale/publique de `gamma_c`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Historique des versions (RMSE de validation locale)

Chaque point est une hypothèse testée ; les annotations donnent l'idée
principale. Les versions neutres comptent autant que les gains : elles
délimitent ce qui est prédictible.

In [ ]:
versions = ["v4", "v5.1", "v9", "v10", "v11", "v12", "v13", "v14", "v15"]
local =    [379.8, 375.4, 371.1, 368.2, 348.4, 346.4, 345.2, 345.9, 345.0]
labels = ["melange probabiliste", "TE hors-fold", "contexte arrivees",
          "panne de reporting", "branche Rome", "ensembling 5 seeds",
          "signaux par avion", "temps non entrave", "file a l'off-block"]

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.plot(versions, local, marker="o", color="#3B82F6")
for x, y, l in zip(versions, local, labels):
    ax.annotate(l, (x, y), textcoords="offset points", xytext=(4, 7), fontsize=8)
ax.set_ylabel("RMSE validation (jan + juil 2025), s")
ax.set_title("Rendements decroissants : -35 s en 11 versions, dont -20 s pour la branche Rome\n"
             "(retiree ensuite : -2.45 s au leaderboard, voir versions.md §3.2)")
plt.tight_layout(); plt.show()

## Calibration locale vs publique de `gamma_c`

Quatre soumissions dédiées (v1 à v4) : la courbe publique est parallèle à la
courbe locale, décalage constant d'environ 32 s. La validation locale est un
proxy fiable pour les propriétés physiques du problème.

In [ ]:
gammas = [0.5, 0.6, 0.75, 1.0]
loc =    [389.8, 379.4, 380.1, 391.0]
pub =    [358.85, 347.80, 347.59, 355.16]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(gammas, loc, marker="o", label="validation locale (jan+juil 2025)", color="#3B82F6")
ax.plot(gammas, pub, marker="s", label="leaderboard (jan+juil 2026)", color="#10B981")
for g, a, b in zip(gammas, loc, pub):
    ax.annotate(f"{a - b:+.0f} s", ((g), (a + b) / 2), fontsize=8, ha="center")
ax.set_xlabel("gamma_c"); ax.set_ylabel("RMSE (s)")
ax.set_title("Le decalage local/public est constant : les gains locaux se transposent")
ax.legend()
plt.tight_layout(); plt.show()

## Scores publics

| Soumission | Pipeline | Rome | Score |
|---|---|---|---|
| v1-v4 | v5.1, balayage gamma | non | 347,59 - 358,85 |
| v7 | v11 | oui | 341,33 |
| v10 | v13 | non | **337,95** |
| v11 | v13 | oui | 340,40 |
| v15 | v15 | oui | 338,02 |

La comparaison v10/v11 (mêmes modèles, seule la branche Rome diffère) mesure
directement le coût du sur-apprentissage d'une distribution de bugs :
**−2,45 s** pour une branche qui gagnait +19,8 s en local.